[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C3/notebooks/samostatne.ipynb)

# Dva léky a druhé omezení — samostatná úloha

**Pracujete sami.** Úloha z tabule dostane druhé omezení:
lék B účinkuje až od **1,5 mg**.

$$
\begin{aligned}
\text{minimize}\quad & (x_1-4)^2 + (x_2-2)^2\\
\text{subject to}\quad & x_1 + x_2 \le 4 && \text{kidney load}\\
& x_2 \ge 1{,}5 && \text{minimal dose of B}
\end{aligned}
$$

## Postup

Tip, která omezení budou aktivní, máte z hlasování na tabuli.

1. Doplňte model a vyřešte ho *(díra 1)*.
2. Ověřte KKT *(díra 2)*. Pozor na znaménko u $x_2 \ge 1{,}5$.
3. Odpovězte slovy: je úloha konvexní? Co znamená $\mu_2$?

---

*Čísla, se kterými se notebook porovnává, počítá skript [`kod/uloha_samostatne.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/uloha_samostatne.py).*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import numpy as np

CIL = np.array([4.0, 2.0])   # ideální dávky léků A a B [mg]
LIMIT = 4.0                  # x1 + x2 <= 4 mg (ledviny)
MIN_B = 1.5                  # x2 >= 1,5 mg (jinak B neúčinkuje)

## Krok 1 — model v CVXPY *(díra 1)*

In [ ]:
# TODO 1: doplňte obě omezení a účelovou funkci (x1 - 4)^2 + (x2 - 2)^2.
x = cp.Variable(2)
ledviny = ...    # x1 + x2 <= 4
min_b = ...      # x2 >= 1,5
omezeni = [ledviny, min_b]
uloha = cp.Problem(cp.Minimize(...), omezeni)   # nápověda: cp.sum_squares(...)

In [ ]:
uloha.solve(solver=cp.CLARABEL)
x_opt = x.value
x1, x2 = x_opt
mu1, mu2 = ledviny.dual_value, min_b.dual_value

print("konvexní (DCP)?", uloha.is_dcp())
print(f"x* = ({x1:.3f}; {x2:.3f}), f* = {uloha.value:.3f}")
print(f"mu = ({mu1:.3f}; {mu2:.3f})")
assert np.allclose(x_opt, [2.5, 1.5], atol=1e-5) and abs(uloha.value - 2.5) < 1e-5
assert np.allclose([mu1, mu2], [3.0, 2.0], atol=1e-4)

## Krok 2 — KKT ručně *(díra 2)*

Tvar ze třetí přednášky: $g_i(\mathbf x) \le 0$ a
$\nabla f + \mu_1\nabla g_1 + \mu_2\nabla g_2 = \mathbf 0$,
$\mu_i\,g_i = 0$, $\mu_i \ge 0$.

In [ ]:
# TODO 2: obě omezení přepište do tvaru g_i(x) <= 0 a doplňte g2 a jeho gradient.
# mu2 z CVXPY je už kladné, otáčíte jen g2 a jeho gradient
g1 = x1 + x2 - LIMIT
g2 = ...                         # x2 >= 1,5  ->  ... <= 0
grad_f = 2 * (x_opt - CIL)
grad_g1 = np.array([1.0, 1.0])
grad_g2 = ...                    # np.array([..., ...])

stacionarita = grad_f + mu1 * grad_g1 + mu2 * grad_g2
komplementarita = ...            # np.array([mu1 * g1, mu2 * g2])

In [ ]:
print("stacionarita    ", np.round(stacionarita, 6))
print("komplementarita ", np.round(komplementarita, 6))
assert np.allclose(stacionarita, 0, atol=1e-4), "gradient g2 má špatné znaménko?"
assert np.allclose(komplementarita, 0, atol=1e-4)
assert g1 <= 1e-6 and g2 <= 1e-6 and mu1 >= 0 and mu2 >= 0
print("KKT splněny")

## Krok 3 — odpovězte slovy

1. **Je úloha konvexní?** Jak to poznáte bez `is_dcp()`?
2. **Co znamená $\mu_2 = 2$** pro lékaře? Ověřte přepočtem v další buňce.

**TODO:** 1. … 2. …

In [ ]:
# stejná úloha, jen minimální dávka B zmírněná o 0,1 mg (ledviny zůstávají)
zmirnena = cp.Problem(uloha.objective, [ledviny, x[1] >= MIN_B - 0.1])
zmirnena.solve(solver=cp.CLARABEL)
print(f"x2 >= 1,4: f* = {zmirnena.value:.3f}, úspora {uloha.value - zmirnena.value:.3f}")